In [8]:
from datasets import load_dataset
# Load CoNLL-2003 dataset for POS tagging & Chunking
dataset = load_dataset("conll2003", trust_remote_code=True)
print(dataset)
print("\nLabel Categories (POS Tags):", dataset["train"].features["pos_tags"].feature.names)
print("Label Categories (Chunk Tags):", dataset["train"].features["chunk_tags"].feature.names)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Label Categories (POS Tags): ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Label Categories (Chunk Tags): ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [9]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    pos_labels_list, chunk_labels_list = [], []
    for i, (pos_label, chunk_label) in enumerate(zip(examples["pos_tags"], examples["chunk_tags"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        prev_word_idx = None
        pos_ids, chunk_ids = [], []
        for word_idx in word_ids:
            if word_idx is None:
                pos_ids.append(-100)
                chunk_ids.append(-100)
            elif word_idx != prev_word_idx:
                pos_ids.append(pos_label[word_idx])
                chunk_ids.append(chunk_label[word_idx])
            else:
                pos_ids.append(-100)
                chunk_ids.append(-100)
            prev_word_idx = word_idx
        pos_labels_list.append(pos_ids)
        chunk_labels_list.append(chunk_ids)
    tokenized_inputs["pos_labels"] = pos_labels_list
    tokenized_inputs["chunk_labels"] = chunk_labels_list
    return tokenized_inputs
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)
print("Tokenization Complete!")

Map: 100%|██████████| 3453/3453 [00:01<00:00, 2770.31 examples/s]

Tokenization Complete!


In [10]:
from transformers import AutoModelForTokenClassification
# POS Model
pos_labels = dataset["train"].features["pos_tags"].feature.names
pos_model = AutoModelForTokenClassification.from_pretrained("distilbert-base-uncased", num_labels=len(pos_labels))
# Chunk Model
chunk_labels = dataset["train"].features["chunk_tags"].feature.names
chunk_model = AutoModelForTokenClassification.from_pretrained("distilbert-base-uncased", num_labels=len(chunk_labels))
print("Models Loaded!")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2136.51it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2832.94it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  

Models Loaded!


In [11]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./pos_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,  # small epoch for fast run
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)
print("Training Arguments Ready!")

Training Arguments Ready!


In [12]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score
def compute_metrics_pos(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds = [[pos_labels[p] for (p,l) in zip(pred, lab) if l!=-100] for pred, lab in zip(predictions, labels)]
    true_labels = [[pos_labels[l] for (p,l) in zip(pred, lab) if l!=-100] for pred, lab in zip(predictions, labels)]
    return {"precision": precision_score(true_labels, true_preds),
            "recall": recall_score(true_labels, true_preds),
            "f1": f1_score(true_labels, true_preds)}
def compute_metrics_chunk(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds = [[chunk_labels[p] for (p,l) in zip(pred, lab) if l!=-100] for pred, lab in zip(predictions, labels)]
    true_labels = [[chunk_labels[l] for (p,l) in zip(pred, lab) if l!=-100] for pred, lab in zip(predictions, labels)]
    return {"precision": precision_score(true_labels, true_preds),
            "recall": recall_score(true_labels, true_preds),
            "f1": f1_score(true_labels, true_preds)}
print("Metric functions ready!")

Metric functions ready!


In [26]:
from transformers import Trainer

# Use small subset for CPU
small_train = tokenized_dataset["train"].select(range(500))
small_eval  = tokenized_dataset["validation"].select(range(100))

pos_trainer = Trainer(
    model=pos_model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("POS Trainer Ready!")

POS Trainer Ready!


In [28]:
from transformers import AutoModelForTokenClassification

pos_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(tokenized_dataset["train"].features["pos_tags"].feature.names)
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4769.67it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

# FAST version — no shuffle
small_train = tokenized_dataset["train"].select(range(200))
small_eval  = tokenized_dataset["validation"].select(range(50))

print("Dataset Ready!")

Dataset Ready!


In [35]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=64
    )

    labels = []

    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels   # ✅ IMPORTANT
    return tokenized_inputs

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print("Tokenization Finished")

Map: 100%|██████████| 3453/3453 [00:00<00:00, 3671.86 examples/s]

Tokenization Finished


In [36]:
small_train = tokenized_dataset["train"].select(range(200))
small_eval  = tokenized_dataset["validation"].select(range(50))

print("Small dataset ready")

Small dataset ready


In [37]:
from transformers import AutoModelForTokenClassification

num_labels = len(dataset["train"].features["pos_tags"].feature.names)

pos_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

print("Model Loaded")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3148.66it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded


In [38]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./pos_model",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=1,   # FAST

    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

print("Training args ready")

Training args ready


In [39]:
from transformers import Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=pos_model,
    args=training_args,

    train_dataset=small_train,
    eval_dataset=small_eval,

    data_collator=data_collator
)

print("Trainer Ready")

Trainer Ready


In [40]:
trainer.train()

Step,Training Loss
10,3.436531
20,2.780200


TrainOutput(global_step=25, training_loss=2.997253112792969, metrics={'train_runtime': 30.5002, 'train_samples_per_second': 6.557, 'train_steps_per_second': 0.82, 'total_flos': 3268985164800.0, 'train_loss': 2.997253112792969, 'epoch': 1.0})

In [41]:
import numpy as np

sentence = "John works at Google in California".split()

inputs = tokenizer(
    sentence,
    is_split_into_words=True,
    return_tensors="pt"
)

outputs = pos_model(**inputs)

predictions = np.argmax(
    outputs.logits.detach().numpy(),
    axis=-1
)

print("Prediction IDs:", predictions)

Prediction IDs: [[22 22 22 22 22 22 22  7]]


In [46]:
label_names = dataset["train"].features["pos_tags"].feature.names

predicted_tags = [
    label_names[p]
    for p in predictions[0]
]

print("Tokens:", sentence)
print("Predicted Tags:", predicted_tags)

Tokens: ['John', 'works', 'at', 'Google', 'in', 'California']
Predicted Tags: ['NNP', 'NNP', 'NNP', 'NNP', 'NNP', 'NNP', 'NNP', '.']


In [47]:
trainer.save_model("./final_pos_model")

tokenizer.save_pretrained("./final_pos_model")

print("Model Saved Successfully!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]

Model Saved Successfully!
